## Batch Processing with the Claude Messages API

### Installing Libraries

In [ ]:
%pip install anthropic python-dotenv

### Setting up the Environment

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
claude_model = os.getenv("CLAUDE_MODEL")

### Creating the Anthropic Client

In [ ]:
import anthropic

client = anthropic.Anthropic(api_key=anthropic_api_key)

### Creating Batch User Prompts

In [ ]:
user_prompt_1 = """
Analyze the following customer review.

Review:
"The received chess set is of good quality, but the packaging was damaged upon arrival.
The pieces are well-crafted and the board is sturdy. Overall, I am satisfied with the
product, but the shipping could be improved."

Return:
- Sentiment
- Confidence
- Once sentence summary
"""

user_prompt_2 = """
Analyze the following customer review.

Review:
"Though the chess pieces are well-crafted but the there is a minor defect on the board.
When the board is unfolded, the uneven hinge causes the board to not lay flat.
Overall, I am disappointed with the product and would not recommend it."

Return:
- Sentiment
- Confidence
- Once sentence summary
"""

user_prompt_3 = """
Analyze the following customer review.

Review:
"My daughters love the chess set. The chess pieces are beautifully designed and
the board is of excellent quality."

Return:
- Sentiment
- Confidence
- Once sentence summary
"""

### Calling the Messages API as Batch

In [ ]:
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request

message_batch = client.messages.batches.create(
    requests=[
        Request(
            custom_id="review-001",
            params=MessageCreateParamsNonStreaming(
                model=claude_model,
                max_tokens=512,
                system="You are a helpful assistant that analyzes customer reviews.",
                messages=[
                    {
                        "role": "user",
                        "content": user_prompt_1
                    }
                ]
            )
        ),
        Request(
            custom_id="review-002",
            params=MessageCreateParamsNonStreaming(
                model=claude_model,
                max_tokens=512,
                system="You are a helpful assistant that analyzes customer reviews.",
                messages=[
                    {
                        "role": "user",
                        "content": user_prompt_2
                    }
                ]
            )
        ),
        Request(
            custom_id="review-003",
            params=MessageCreateParamsNonStreaming(
                model=claude_model,
                max_tokens=512,
                system="You are a helpful assistant that analyzes customer reviews.",
                messages=[
                    {
                        "role": "user",
                        "content": user_prompt_3
                    }
                ]
            )
        )
    ]
)

print(message_batch)

# Storing the message batch ID for later use and polling the batch for results
message_batch_id = message_batch.id

### Polling the Batch for Results

In [ ]:
import time

while True:
    message_batch = client.messages.batches.retrieve(message_batch_id)
    if message_batch.processing_status == "ended":
        for result in client.messages.batches.results(
            message_batch_id,
        ):
            match result.result.type:
                case "succeeded":
                    print(f"Success! {result.custom_id}")
                case "errored":
                    if result.result.error.error.type == "invalid_request_error":
                        # Request body must be fixed before re-sending request
                        print(f"Validation error {result.custom_id}")
                    else:
                        # Request can be retried directly
                        print(f"Server error {result.custom_id}")
                case "expired":
                    print(f"Request expired {result.custom_id}")
    else:
        print(f"Batch processing status: {message_batch.processing_status}")
        time.sleep(60)  # Wait for 60 seconds before polling again